# Xinference Server (Colab GPU)

Set the runtime to **GPU** (Runtime -> Change runtime type -> GPU), then run all cells.
This boots Xinference and launches an LLM, an embedding model, and a reranker.

The models run on Colab's GPU, not your Mac. Leave this notebook running while you work.

In [ ]:
# 1. Install Xinference (a few minutes)
!pip install "xinference[all]" -q

In [ ]:
# 2. Start the Xinference server in the background
import subprocess, time
subprocess.Popen(
    ["xinference-local", "-H", "0.0.0.0", "--port", "9997"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("Booting server...")
time.sleep(25)
print("Server should be up at http://localhost:9997")

In [ ]:
# 3. Launch three models.
# T4-SAFE CONFIG: a 3B LLM on the Transformers engine fits a free 16GB T4
# alongside the embedder + reranker without running out of memory.
#   - Lighter still? set model_size_in_billions="1_5" (1.5B) or "0_5" (0.5B).
#   - More VRAM (Colab Pro A100/L4)? bump to 7 and/or switch to model_engine="vLLM"
#     for higher throughput on the Stage 03 load test.
from xinference.client import Client
client = Client("http://localhost:9997")

llm_uid = client.launch_model(
    model_name="qwen2.5-instruct",
    model_engine="Transformers",
    model_format="pytorch",
    model_size_in_billions=3,
)
print("LLM:", llm_uid)

embed_uid = client.launch_model(
    model_name="bge-base-en-v1.5", model_type="embedding",
)
print("Embedding:", embed_uid)

rerank_uid = client.launch_model(
    model_name="bge-reranker-base", model_type="rerank",
)
print("Reranker:", rerank_uid)

In [ ]:
# 4. Quick test that the LLM answers
model = client.get_model(llm_uid)
out = model.chat(messages=[{"role": "user", "content": "Say hello in one short line."}])
print(out)

## 5. (Optional) Expose the server to your Mac with an ngrok tunnel

Skip this if you're running the stage code inside the notebook. Use it when you want
the client on your Mac to reach this Colab server.

Get a free authtoken at https://ngrok.com and paste it below.

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

NGROK_AUTHTOKEN = "PASTE_YOUR_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(9997).public_url
print("Public endpoint (put this in your .env as XINFERENCE_ENDPOINT):")
print(public_url)